# Essai : le jeu de base (niveau 0) séparé par la codification de la banque (`PAY_1` ≥ 2), puis résultats accumulés

**Question** : la codification de risque posée par la banque suffit-elle ? Ici, on sépare les clients comme le ferait quelqu'un qui part du fichier sans étude : **en retard en septembre** (`PAY_1` ≥ 2, au moins deux mois de retard selon la banque) d'un côté, **tous les autres** de l'autre, puis on applique la même méthode que pour la séparation par la règle métier (`essai_jeu_corrige_deux_populations.ipynb` : population en incident à M, retard imminent, autres clients). C'est le **pendant naïf** de ce travail : on pourra comparer si la règle métier et tout le travail de nettoyage font mieux que la codification de la banque.

**Données** : `data/creditcard_pret_ingestion.csv` (niveau 0 : nettoyage structurel seul), les **23 colonnes d'origine**, tous les clients du fichier : **le même périmètre que l'étude d'origine** (Yeh et Lien, 2009). `PAY_1` n'y est pas corrigé (codifications 1 de septembre gardées telles quelles ; codifications 2 d'avril à juin non uniformisées).

**Méthode, pour chaque population** : exactement celle de `essai_jeu_corrige.ipynb` (même code, repris tel quel) : trois variantes (complet, `PAY_n` seuls, sans les `PAY_n`), 4 modèles, boucle de GridSearch sur la précision moyenne avec un seuil de surapprentissage de 7 %, seuil de décision par la **règle marginale** (au plus 1 bon client signalé pour 1 défaut attrapé, à la marge, `TunedThresholdClassifierCV`), validation croisée stratifiée à 5 plis, `random_state=42`, importance par permutation par famille de colonnes.

**Tolérance de la règle marginale, par population** (fixée par le métier, la même que pour la séparation par la règle métier) :
- **`PAY_1` < 2** : au plus **1 bon client signalé pour 1 défaut attrapé**, à la marge ;
- **`PAY_1` ≥ 2** (le pendant naïf de la population en incident et du retard imminent) : un client n'y est considéré comme sain que s'il a **au moins 9 chances sur 10 de l'être**, à la marge ; côté défaut, il reste signalé dès 10 % de risque. Si le modèle ne considère personne comme sain dans ce groupe, c'est qu'il ne peut pas trancher à ce niveau d'exigence.

**Synthèse globale, sur tous les clients du jeu de base** (section 5) : chaque client reçoit la décision de sa population (hors pli). Stratégies comparées, avec le **meilleur modèle de la variante complète** de chaque population (la plus haute précision moyenne hors pli, règle fixée avant) :
1. **règle de la banque seule** : tous les clients `PAY_1` ≥ 2 en défaut, personne d'autre ;
2. **règle de la banque + ML pour les autres** ;
3. **règle de la banque + ML pour les autres sans les `PAY_n`** (le comportement seul) ;
4. **deux ML** : un modèle par population ;
5. **un seul ML** sur tous les clients (`essai_jeu_de_base.ipynb`, lu dans le fichier qu'il enregistre).

Pour chacune : défauts détectés, clients signalés, précision, fausses alertes, taux d'erreur, ratio d'aire ; puis le **comparatif avec l'étude d'origine, sur le même périmètre**.

**Comparaison avec la séparation par la règle métier** (section 6) : les décisions de ce notebook face à celles de `essai_jeu_corrige_deux_populations.ipynb`, **sur les clients communs aux deux** (ceux du niveau 3), pour savoir si la règle métier et le nettoyage font mieux que la codification de la banque.

⚠️ **Le test du ML est inclus** dans ces clients : aucune conclusion de cet essai ne sert à choisir le modèle final. **Ce n'est pas un scénario** : rien n'est écrit dans le fichier de résultats ni dans le journal, aucun modèle n'est enregistré ; les décisions, client par client, sont enregistrées dans `data/ML/comparatif_global/jeu_de_base_deux_populations.csv`.

**Étapes** :
1. chargement et séparation des deux populations ;
2. pipeline, critère et boucle (repris de `essai_jeu_corrige.ipynb`) ;
3. boucle d'optimisation, puis seuil optimisé, pour chaque population et chaque variante ;
4. ce qui porte les prédictions, dans chaque population (importance par famille) ;
5. synthèse globale : résultats accumulés sur tous les clients, face à la règle de la banque seule, à un seul ML et à l'étude d'origine (même périmètre) ;
6. comparaison avec la séparation par la règle métier, sur les clients communs.

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())

COLONNES_ORIGINE = ["LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE"] + [f"PAY_{n}" for n in range(1, 7)] \
    + [f"BILL_AMT{n}" for n in range(1, 7)] + [f"PAY_AMT{n}" for n in range(1, 7)]
donnees = pd.read_csv(root_dir / "data" / "creditcard_pret_ingestion.csv")   # niveau 0 du nettoyage
assert donnees["ID"].is_unique and not donnees[COLONNES_ORIGINE + ["dpnm"]].isna().any().any()
# Séparation par la codification de risque de la banque : en retard en septembre (PAY_1 >= 2, au moins deux mois de retard)
au_ctx = (donnees["PAY_1"] >= 2).to_numpy()
POPULATIONS = {
    "PAY_1 ≥ 2": donnees[au_ctx][["ID"] + COLONNES_ORIGINE + ["dpnm"]].reset_index(drop=True),
    "PAY_1 < 2": donnees[~au_ctx][["ID"] + COLONNES_ORIGINE + ["dpnm"]].reset_index(drop=True),
}
# Tolérance de la règle marginale, par population (même logique que pour la règle métier du jeu corrigé) : PAY_1 >= 2 en joue le rôle,
# un client n'y est considéré comme sain que s'il a au moins 9 chances sur 10 de l'être ; les autres : 1 bon client signalé pour 1 défaut, à la marge
TOLERANCE = {"PAY_1 ≥ 2": 1 / 9, "PAY_1 < 2": 1}
for nom, d in POPULATIONS.items():
    print(f"{nom} : {len(d)} clients, {d['dpnm'].sum()} défauts ({d['dpnm'].mean() * 100:.1f} %) ; "
          f"signalé en défaut dès {TOLERANCE[nom] / (1 + TOLERANCE[nom]):.0%} de risque à la marge")
print(f"Ensemble : {len(donnees)} clients, {donnees['dpnm'].sum()} défauts ({donnees['dpnm'].mean() * 100:.1f} %)")

# Population active : les cellules reprises de essai_jeu_corrige.ipynb travaillent sur « original » et « y_train » (ici, l'ensemble, pour les définitions)
original = donnees[["ID"] + COLONNES_ORIGINE + ["dpnm"]].reset_index(drop=True)
y_train = original["dpnm"].reset_index(drop=True)

PAY_1 ≥ 2 : 3130 clients, 2177 défauts (69.6 %) ; signalé en défaut dès 10% de risque à la marge
PAY_1 < 2 : 26870 clients, 4459 défauts (16.6 %) ; signalé en défaut dès 50% de risque à la marge
Ensemble : 30000 clients, 6636 défauts (22.1 %)


## 2. Variantes, pipeline, critère et boucle : repris de `essai_jeu_corrige.ipynb`

In [3]:
# ==============================================================================
# VARIABLES : les 23 variables d'origine, rangées selon leur encodage (celui des scénarios, méthode v9)
# ==============================================================================
PAY = [f"PAY_{n}" for n in range(1, 7)]

# PAY_n : catégories, une colonne par codification (v9)
# PAY_AMTn et LIMIT_BAL : montants très étirés, logarithme (plafond au logarithme : v9)
# BILL_AMTn : factures, logarithme signé, car elles peuvent être négatives (v8)
# SEX, EDUCATION, MARRIAGE : catégories sans ordre, une colonne par catégorie (regroupées dès le niveau 0)
# AGE : nombre, mis à l'échelle
TYPES_ORIGINE = {
    "num": ["AGE"],
    "log": [f"PAY_AMT{n}" for n in range(1, 7)]+["LIMIT_BAL"],
    "log_signe": [f"BILL_AMT{n}" for n in range(1, 7)],
    "cat": ["SEX", "EDUCATION", "MARRIAGE"] + PAY,
}

# Les trois variantes : colonnes gardées, chacune avec son encodage d'origine
VARIANTES = {
    "Complet": lambda c: True,
    "PAY_n seuls": lambda c: c in PAY,
    "Sans les PAY_n": lambda c: c not in PAY,
}
TYPES_VARIANTE = {v: {t: [c for c in cols if garder(c)] for t, cols in TYPES_ORIGINE.items()} for v, garder in VARIANTES.items()}
COLONNES_VARIANTE = {v: [c for cols in types.values() for c in cols] for v, types in TYPES_VARIANTE.items()}

# Contrôles : colonnes présentes, ni identifiant ni cible, les 23 variables couvertes par la variante complète
assert sorted(COLONNES_VARIANTE["Complet"]) == sorted(c for c in original.columns if c not in ("ID", "dpnm"))
assert sorted(COLONNES_VARIANTE["PAY_n seuls"] + COLONNES_VARIANTE["Sans les PAY_n"]) == sorted(COLONNES_VARIANTE["Complet"])
assert (original[TYPES_ORIGINE["log"]] >= 0).all().all(), "Le logarithme ne s'applique qu'à des montants positifs ou nuls"

display(pd.DataFrame({
    "Variante": list(VARIANTES),
    "Nombre de variables": [len(COLONNES_VARIANTE[v]) for v in VARIANTES],
    "Variables": [", ".join(COLONNES_VARIANTE[v]) for v in VARIANTES],
}).set_index("Variante"))

,Nombre de variables,Variables
Variante,,
Complet,23,"AGE, PAY_AMT1, PAY_AMT2, PAY_AMT3, PAY_AMT4, P..."
PAY_n seuls,6,"PAY_1, PAY_2, PAY_3, PAY_4, PAY_5, PAY_6"
Sans les PAY_n,17,"AGE, PAY_AMT1, PAY_AMT2, PAY_AMT3, PAY_AMT4, P..."


In [4]:
from catboost import CatBoostClassifier
from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", None)   # textes affichés en entier

# Pondération des défauts pour CatBoost (part des non-défauts / part des défauts)
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

# Transformation de chaque type de colonne (un nouvel objet à chaque appel)
TRANSFORMATIONS = {
    "num": lambda: StandardScaler(),
    "log": lambda: make_pipeline(FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler()),
    "log_signe": lambda: make_pipeline(FunctionTransformer(np.arcsinh, feature_names_out="one-to-one"), StandardScaler()),
    "cat": lambda: OneHotEncoder(sparse_output=False, handle_unknown="ignore"),
}

def preprocesseur(variante):
    # Une branche par type de colonne, seulement pour les types présents dans la variante
    return ColumnTransformer(transformers=[(nom, TRANSFORMATIONS[nom](), colonnes)
                                           for nom, colonnes in TYPES_VARIANTE[variante].items() if colonnes])

# Critère optimisé par la boucle : la précision moyenne (aire sous la courbe précision-rappel), sur tous les rappels
CRITERE = "average_precision"
scorers = {
    "average_precision": "average_precision",
    "roc_auc": "roc_auc",
}

# Rappel minimal des scénarios (60 %) : sert seulement à la comparaison avec les scénarios (section 6)
RAPPEL_MINIMAL = 0.60

def seuil_pour_rappel(y, proba, rappel=RAPPEL_MINIMAL):
    # Seuil le plus haut qui détecte au moins la part voulue des défauts
    proba_defauts = np.sort(proba[np.asarray(y) == 1])[::-1]
    return proba_defauts[int(np.ceil(rappel * len(proba_defauts))) - 1]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Modèles de l'essai : les meilleurs des scénarios, avec leurs réglages de la première itération
MODELES_A_TESTER = ["LogisticRegression", "MLPClassifier", "RandomForest", "CatBoost"]

def modeles_de_base(variante):
    prep = preprocesseur(variante)
    tous = {
        "LogisticRegression": make_pipeline(prep, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
        "MLPClassifier": make_pipeline(prep, MLPClassifier(hidden_layer_sizes=(50, 25), max_iter=500, random_state=42, early_stopping=True)),
        "RandomForest": make_pipeline(prep, RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)),
        "CatBoost": make_pipeline(prep, CatBoostClassifier(scale_pos_weight=scale_pos, random_state=42, verbose=0, thread_count=-1, allow_writing_files=False)),
    }
    return {nom: tous[nom] for nom in MODELES_A_TESTER}

# ------------------------------------------------------------------------------
# Alertes et tableaux de scores (ceux des scénarios)
SEUIL_SURAPPRENTISSAGE = 0.07   # écart relatif train - val du critère : 7 % pour cet essai (5 % dans les scénarios) ; le double : surapprentissage fort
SEUIL_INSTABILITE = 0.02        # écart-type du critère sur les plis au-delà duquel on alerte (alerte seule, pas une contrainte)
METRIQUES = {"average_precision": "Précision moyenne", "roc_auc": "ROC AUC"}
LIBELLE_SCORE = "Précision des défauts prédits"
TAUX_DEFAUT = y_train.mean()    # précision, et précision moyenne, d'un tri au hasard

def apprend(score_val, std_val):
    # Plancher contre le sous-apprentissage : le critère doit dépasser le hasard (taux de défaut) de plus d'un écart-type
    return score_val - TAUX_DEFAUT > std_val

def alerte(ecart):
    if ecart > 2 * SEUIL_SURAPPRENTISSAGE:
        return "⚠️⚠️ SURAPPRENTISSAGE FORT"
    if ecart > SEUIL_SURAPPRENTISSAGE:
        return "⚠️ SURAPPRENTISSAGE"
    return "✅ OK"

def ligne_scores(nom, moy_val, std_val, moy_train):
    # Une ligne de tableau : chaque métrique en entraînement et en validation, le critère, son écart-type et l'alerte
    ligne = {"Modèle": nom}
    for cle, libelle in METRIQUES.items():
        ligne[f"{libelle} (train)"] = moy_train[cle]
        ligne[f"{libelle} (val)"] = moy_val[cle]
    ligne["Critère (val)"], ligne["Critère (train)"] = moy_val[CRITERE], moy_train[CRITERE]
    ligne["Écart-type critère (val)"] = std_val
    ligne["Écart train - val"] = moy_train[CRITERE] - moy_val[CRITERE]
    ligne["Écart relatif"] = ligne["Écart train - val"] / moy_val[CRITERE]
    ligne["Apprend"] = apprend(moy_val[CRITERE], std_val)
    ligne["Alerte"] = alerte(ligne["Écart relatif"]) if ligne["Apprend"] else "⚠️ N'APPREND RIEN (niveau du hasard)"
    if std_val > SEUIL_INSTABILITE:
        ligne["Alerte"] += " ; ⚠️ INSTABLE"
    return ligne

def pct(serie):
    return (serie * 100).round(1)

def parametres(params):
    # Paramètres sur une ligne : « depth 4, 5, 6 ; l2_leaf_reg 15 »
    return " ; ".join(f"{p} {', '.join(map(str, v)) if isinstance(v, list) else v}" for p, v in params.items())

def montrer(df, titre):
    display(Markdown(f"**{titre}**"))
    display(df.set_index(df.columns[0]))

In [5]:
from itertools import product
from sklearn.model_selection import GridSearchCV

TOURS_SECURITE = 15   # garde-fou contre une boucle sans fin : la boucle s'arrête d'elle-même avant ; s'il est atteint, revoir les grilles ou les bornes

# Grilles du premier tour : celles des scénarios
grilles = {
    "LogisticRegression": {"logisticregression__C": [0.01, 0.1, 1.0, 10.0]},
    "MLPClassifier": {"mlpclassifier__hidden_layer_sizes": [(32, 16), (64, 32), (32,)], "mlpclassifier__alpha": [0.01, 0.1, 1.0]},
    "RandomForest": {"randomforestclassifier__max_depth": [6, 8, 10], "randomforestclassifier__min_samples_leaf": [20, 50, 100], "randomforestclassifier__n_estimators": [200, 300]},
    "CatBoost": {"catboostclassifier__depth": [4, 6], "catboostclassifier__l2_leaf_reg": [5, 10, 15], "catboostclassifier__learning_rate": [0.03, 0.05],
                 "catboostclassifier__iterations": [1000]},   # 1 000 arbres : valeur par défaut, réduite par la boucle en cas de surapprentissage
}

# Comment faire évoluer chaque paramètre numérique d'un tour à l'autre (ceux des scénarios) :
# - simplifier : sens qui réduit le surapprentissage (+1 = augmenter, -1 = diminuer) ;
# - pas : écart entre deux valeurs (« x » = multiplier, « + » = ajouter) ;
# - bornes : valeurs minimale et maximale autorisées.
# Les paramètres absents de ce tableau (architecture du MLP, nombre d'arbres du RandomForest) sont fixés à leur meilleure valeur après le premier tour.
REGLAGES_PARAMETRES = {
    "logisticregression__C":                    {"simplifier": -1, "pas": ("x", 3),   "bornes": (1e-4, 1e4),  "entier": False},
    "mlpclassifier__alpha":                     {"simplifier": +1, "pas": ("x", 3),   "bornes": (1e-5, 100),  "entier": False},
    "randomforestclassifier__max_depth":        {"simplifier": -1, "pas": ("+", 2),   "bornes": (2, 30),      "entier": True},
    "randomforestclassifier__min_samples_leaf": {"simplifier": +1, "pas": ("x", 1.5), "bornes": (1, 2000),    "entier": True},
    "catboostclassifier__depth":                {"simplifier": -1, "pas": ("+", 1),   "bornes": (2, 10),      "entier": True},
    "catboostclassifier__l2_leaf_reg":          {"simplifier": +1, "pas": ("x", 1.5), "bornes": (1, 200),     "entier": False},
    "catboostclassifier__learning_rate":        {"simplifier": -1, "pas": ("x", 1.5), "bornes": (0.003, 0.3), "entier": False},
    "catboostclassifier__iterations":           {"simplifier": -1, "pas": ("x", 1.5), "bornes": (100, 3000),  "entier": True},
}

def nom_court(parametre):
    return parametre.split("__")[-1]

def arrondir(valeur, entier):
    return int(round(valeur)) if entier else float(f"{valeur:.3g}")

def voisin(valeur, reglage, sens):
    # Valeur suivante dans un sens (+1 = plus grande, -1 = plus petite), dans les bornes
    mode, pas = reglage["pas"]
    if mode == "x":
        nouvelle = valeur * pas if sens > 0 else valeur / pas
    else:
        nouvelle = valeur + pas if sens > 0 else valeur - pas
    bas, haut = reglage["bornes"]
    nouvelle = arrondir(min(max(nouvelle, bas), haut), reglage["entier"])
    return None if nouvelle == valeur else nouvelle

def choisir_index(r):
    # Choix sous contrainte, parmi les réglages qui battent le hasard : le meilleur critère sans surapprentissage ;
    # sinon le moins surappris ; si aucun ne bat le hasard, le meilleur score
    val, std = r[f"mean_test_{CRITERE}"], r[f"std_test_{CRITERE}"]
    ecarts = (r[f"mean_train_{CRITERE}"] - val) / val   # écarts relatifs
    utiles = apprend(val, std)
    sous_seuil = np.where(utiles & (ecarts <= SEUIL_SURAPPRENTISSAGE))[0]
    if len(sous_seuil):
        return int(sous_seuil[np.argmax(val[sous_seuil])])
    if utiles.any():
        idx = np.where(utiles)[0]
        return int(idx[np.argmin(ecarts[idx])])
    return int(np.argmax(val))

def bords_de_grille(grille, meilleurs):
    # Paramètres numériques dont la meilleure valeur est la plus petite ou la plus grande de la grille (+1 = maximum, -1 = minimum)
    bords = {}
    for p, valeurs in grille.items():
        if p in REGLAGES_PARAMETRES and len(valeurs) > 1:
            if meilleurs[p] == max(valeurs):
                bords[p] = +1
            elif meilleurs[p] == min(valeurs):
                bords[p] = -1
    return bords

def grille_suivante(grille, meilleurs, bords, surapprend):
    # Grille du tour suivant et explication des corrections ; None si plus rien ne peut bouger
    nouvelle, actions = {}, []
    for p, valeurs in grille.items():
        v = meilleurs[p]
        reglage = REGLAGES_PARAMETRES.get(p)
        if reglage is None:
            nouvelle[p] = [v]                            # catégorie ou paramètre sans effet : fixé
            continue
        sens = None
        if surapprend and reglage["simplifier"] != 0:
            sens = reglage["simplifier"]                 # pousser vers un modèle plus simple
            raison = "surapprentissage"
        elif p in bords:
            sens = bords[p]                              # prolonger la grille du côté du bord
            raison = "bord de grille"
        if sens is None:
            nouvelle[p] = [v]
            continue
        v1 = voisin(v, reglage, sens)
        v2 = voisin(v1, reglage, sens) if v1 is not None else None
        valeurs_tour = [x for x in (v, v1, v2) if x is not None]
        nouvelle[p] = sorted(set(valeurs_tour))
        if len(nouvelle[p]) > 1:
            actions.append(f"{nom_court(p)} {v} → {nouvelle[p]} ({raison})")
    if not actions:
        return None, ["bornes atteintes : la grille ne peut plus bouger"]
    return nouvelle, actions

def lancer_grille(nom, modele, grille, X):
    grid = GridSearchCV(modele, grille, cv=cv, scoring=scorers, refit=choisir_index, return_train_score=True, n_jobs=-1)
    grid.fit(X, y_train)
    r, i = grid.cv_results_, grid.best_index_
    ligne = ligne_scores(nom,
                         {k: r[f"mean_test_{k}"][i] for k in METRIQUES},
                         r[f"std_test_{CRITERE}"][i],
                         {k: r[f"mean_train_{k}"][i] for k in METRIQUES})
    ligne["Meilleurs paramètres"] = {nom_court(k): v for k, v in grid.best_params_.items()}
    return grid, ligne

def cle_reglage(params):
    # Identifiant d'une combinaison de paramètres, pour savoir si elle a déjà été testée
    return str(sorted(params.items()))

def reglages_nouveaux(grille, deja_testes):
    # Combinaisons de la grille qui n'ont encore été testées à aucun tour
    combinaisons = [dict(zip(grille, valeurs)) for valeurs in product(*grille.values())]
    return [c for c in combinaisons if cle_reglage(c) not in deja_testes]

def choisir_tour(tours):
    # Version retenue parmi les tours joués : d'abord ceux qui battent le hasard et sans surapprentissage
    # (sinon ceux qui battent le hasard, sinon tous) ; parmi eux, ceux à moins d'un écart-type du meilleur score
    # (à égalité avec lui) ; et parmi ces derniers, celui qui surapprend le moins
    for garder in (lambda l: l["Apprend"] and l["Écart relatif"] <= SEUIL_SURAPPRENTISSAGE, lambda l: l["Apprend"], lambda l: True):
        groupe = [t for t in tours if garder(t[1])]
        if groupe:
            break
    meilleur = max(groupe, key=lambda t: t[1]["Critère (val)"])[1]
    egaux = [t for t in groupe if meilleur["Critère (val)"] - t[1]["Critère (val)"] <= meilleur["Écart-type critère (val)"]]
    return min(egaux, key=lambda t: t[1]["Écart relatif"])

def boucle(variante):
    # Boucle des scénarios, pour un jeu de variables : version retenue de chaque modèle, lignes de scores et historique
    X = original[COLONNES_VARIANTE[variante]].reset_index(drop=True)
    best_models, lignes_retenues, historique = {}, [], []
    for nom, modele in modeles_de_base(variante).items():
        grille, retenu, fin, deja_testes, reglage_precedent, tours = grilles[nom], None, None, set(), None, []
        for tour in range(1, TOURS_SECURITE + 1):
            grid, ligne = lancer_grille(nom, modele, grille, X)
            ligne["Tour"] = tour
            deja_testes |= {cle_reglage(c) for c in grid.cv_results_["params"]}
            surapprend = ligne["Écart relatif"] > SEUIL_SURAPPRENTISSAGE
            bords = bords_de_grille(grille, grid.best_params_)
            precedent = retenu
            tours.append((grid, ligne))
            retenu = choisir_tour(tours)
            grille_tour = {nom_court(p): v for p, v in grille.items()}
            reglage = ligne["Meilleurs paramètres"]
            # Stagnation : seulement entre deux tours sans surapprentissage
            stagne = (precedent is not None and not surapprend
                      and precedent[1]["Écart relatif"] <= SEUIL_SURAPPRENTISSAGE
                      and ligne["Critère (val)"] - precedent[1]["Critère (val)"] <= precedent[1]["Écart-type critère (val)"])
            if not ligne["Apprend"]:
                fin = "sous-apprentissage : le réglage choisi ne bat plus le hasard"
            elif not surapprend and not bords:
                fin = "optimisé : ni surapprentissage ni paramètre au bord"
            elif reglage == reglage_precedent:
                fin = "convergé : le réglage choisi ne change plus"
            elif stagne:
                fin = "stagnation : gain plus petit que l'écart-type, entre deux tours sans surapprentissage"
            elif tour == TOURS_SECURITE:
                fin = "⚠️ garde-fou atteint : grilles ou bornes à revoir"
            else:
                grille, actions = grille_suivante(grille, grid.best_params_, bords, surapprend)
                if grille is None:
                    fin = actions[0]
                elif not reglages_nouveaux(grille, deja_testes):
                    fin = "convergé : la grille suivante ne contient aucun réglage nouveau"
            reglage_precedent = reglage
            historique.append({
                "Modèle": nom, "Tour": tour, "Grille testée": grille_tour, "Réglage choisi": reglage,
                "Critère (val)": ligne["Critère (val)"], "Écart-type": ligne["Écart-type critère (val)"],
                "Écart relatif": ligne["Écart relatif"], "Alerte": ligne["Alerte"],
                "Suite": fin if fin else "; ".join(actions),
            })
            if fin:
                break
        best_models[nom] = retenu[0].best_estimator_
        lignes_retenues.append(dict(retenu[1], **{"Arrêt": fin, "Tours joués": tour}))
        print(f"  {nom} : {tour} tour(s), {fin} ; version retenue : tour {retenu[1]['Tour']}")
    return best_models, pd.DataFrame(lignes_retenues), pd.DataFrame(historique)

## 3. Boucle d'optimisation et seuil optimisé, pour chaque population

Pour chaque population : la boucle règle les 4 modèles dans chaque variante (une ligne par modèle pendant le calcul), puis chaque version retenue est réentraînée sur chaque pli avec son **seuil par la règle marginale** (validation croisée interne), et jugée sur le pli de validation. Le tableau de chaque population donne, en validation (moyenne ± écart-type sur les 5 plis) : le réglage retenu, la précision moyenne et le ROC AUC (sans seuil), puis, au seuil optimisé, le rappel, la précision des défauts prédits, la précision marginale (contrôle de la règle, visée 50 %) et la part des clients signalés.

In [6]:
import time
from sklearn.base import clone
from sklearn.metrics import average_precision_score, make_scorer, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import TunedThresholdClassifierCV

cv_interne = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)   # plis internes, pour le choix du seuil
PART_MARGE = 0.10

def gain_net(y_vrai, y_pred, tolerance):
    # Défauts détectés moins « tolérance » × bons clients signalés : maximal au seuil où le risque des derniers clients signalés
    # vaut tolerance / (1 + tolerance) (50 % pour une tolérance de 1, 10 % pour 1/9)
    y_vrai, y_pred = np.asarray(y_vrai), np.asarray(y_pred).astype(bool)
    return ((y_pred & (y_vrai == 1)).sum() - tolerance * (y_pred & (y_vrai == 0)).sum()) / len(y_vrai)

def precision_marginale(y_vrai, proba, signales):
    idx = np.where(signales)[0]
    if len(idx) == 0:
        return np.nan
    derniers = idx[np.argsort(proba[idx])[:max(1, int(np.ceil(PART_MARGE * len(idx))))]]
    return y_vrai[derniers].mean()

def moy_et(valeurs, en_pct=False):
    facteur = 100 if en_pct else 1
    return f"{np.nanmean(valeurs) * facteur:.{1 if en_pct else 3}f} ± {np.nanstd(valeurs) * facteur:.{1 if en_pct else 3}f}"

def activer(population):
    # Rend une population active pour les fonctions reprises de l'essai (elles lisent ces variables globales)
    global original, y_train, scale_pos, TAUX_DEFAUT, y, plis, SCORE_SEUIL
    original = POPULATIONS[population]
    SCORE_SEUIL = make_scorer(gain_net, tolerance=TOLERANCE[population])   # tolérance de la règle marginale propre à la population
    y_train = original["dpnm"].reset_index(drop=True)
    y = y_train.to_numpy()
    scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
    TAUX_DEFAUT = y_train.mean()
    plis = list(cv.split(original, y_train))

ETUDE = {}   # population -> {"probas", "decisions", "mesures", "modeles_plis", "plis", "retenus"}
for population in POPULATIONS:
    activer(population)
    debut = time.time()
    display(Markdown(f"### {population} : {len(original)} clients, taux de défaut {TAUX_DEFAUT * 100:.1f} % ; "
                     f"tolérance {TOLERANCE[population]:.3g} (signalé en défaut dès {TOLERANCE[population] / (1 + TOLERANCE[population]):.0%} de risque à la marge)"))
    probas, decisions, mesures, modeles_plis, retenus = {}, {}, {}, {}, {}
    for variante in VARIANTES:
        print(f"Variante « {variante} »")
        best_models, df_retenus, _ = boucle(variante)
        retenus[variante] = df_retenus.set_index("Modèle")
        X = original[COLONNES_VARIANTE[variante]].reset_index(drop=True)
        for nom, modele in best_models.items():
            p, d, entraines = np.zeros(len(y)), np.zeros(len(y), dtype=int), []
            m_plis = {"Rappel": [], "Précision": [], "Précision marginale": [], "Signalés": [], "Précision moyenne": [], "ROC AUC": [], "Sans tri": 0}
            for entrainement, validation in plis:
                try:
                    seuil_auto = TunedThresholdClassifierCV(clone(modele), scoring=SCORE_SEUIL, cv=cv_interne, n_jobs=-1)
                    seuil_auto.fit(X.iloc[entrainement], y[entrainement])
                    p[validation] = seuil_auto.predict_proba(X.iloc[validation])[:, 1]
                    d[validation] = seuil_auto.predict(X.iloc[validation])
                except ValueError as erreur:
                    if "constant predictions" not in str(erreur):
                        raise
                    # Le modèle donne la même probabilité à tous les clients des plis internes : il ne trie pas, aucun seuil ne se cherche.
                    # Repli : seuil de la règle marginale lu sur la probabilité elle-même (tolérance / (1 + tolérance))
                    seuil_auto = clone(modele).fit(X.iloc[entrainement], y[entrainement])
                    p[validation] = seuil_auto.predict_proba(X.iloc[validation])[:, 1]
                    d[validation] = (p[validation] >= TOLERANCE[population] / (1 + TOLERANCE[population])).astype(int)
                    m_plis["Sans tri"] += 1
                y_val, dec = y[validation], d[validation].astype(bool)
                m_plis["Rappel"].append(recall_score(y_val, dec))
                m_plis["Précision"].append(precision_score(y_val, dec, zero_division=0))
                m_plis["Précision marginale"].append(precision_marginale(y_val, p[validation], dec))
                m_plis["Signalés"].append(dec.mean())
                m_plis["Précision moyenne"].append(average_precision_score(y_val, p[validation]))
                m_plis["ROC AUC"].append(roc_auc_score(y_val, p[validation]))
                entraines.append(seuil_auto)
            probas[(variante, nom)], decisions[(variante, nom)], mesures[(variante, nom)], modeles_plis[(variante, nom)] = p, d, m_plis, entraines
    ETUDE[population] = {"probas": probas, "decisions": decisions, "mesures": mesures, "modeles_plis": modeles_plis, "plis": plis, "retenus": retenus}
    lignes = []
    for (variante, nom), m in mesures.items():
        r = retenus[variante].loc[nom]
        lignes.append({"Variante": variante, "Modèle": nom, "Réglage retenu": parametres(r["Meilleurs paramètres"]),
                       "Écart relatif (%)": round(r["Écart relatif"] * 100, 1),
                       "Précision moyenne": moy_et(m["Précision moyenne"]), "ROC AUC": moy_et(m["ROC AUC"]),
                       "Rappel (%)": moy_et(m["Rappel"], True), "Précision (%)": moy_et(m["Précision"], True),
                       "Précision marginale (%)": moy_et(m["Précision marginale"], True), "Clients signalés (%)": moy_et(m["Signalés"], True),
                       "Plis sans tri": m["Sans tri"]})
    display(Markdown(f"**{population} : détection en validation, au seuil de la règle marginale** (précision marginale visée : "
                     f"{TOLERANCE[population] / (1 + TOLERANCE[population]):.0%} de défauts parmi les derniers signalés ; {(time.time() - debut) / 60:.1f} min)"))
    display(pd.DataFrame(lignes).set_index(["Variante", "Modèle"]))

### PAY_1 ≥ 2 : 3130 clients, taux de défaut 69.6 % ; tolérance 0.111 (signalé en défaut dès 10% de risque à la marge)

Variante « Complet »
  LogisticRegression : 1 tour(s), optimisé : ni surapprentissage ni paramètre au bord ; version retenue : tour 1
  MLPClassifier : 2 tour(s), convergé : le réglage choisi ne change plus ; version retenue : tour 1
  RandomForest : 2 tour(s), convergé : le réglage choisi ne change plus ; version retenue : tour 1
  CatBoost : 3 tour(s), convergé : le réglage choisi ne change plus ; version retenue : tour 2
Variante « PAY_n seuls »
  LogisticRegression : 1 tour(s), optimisé : ni surapprentissage ni paramètre au bord ; version retenue : tour 1
  MLPClassifier : 1 tour(s), optimisé : ni surapprentissage ni paramètre au bord ; version retenue : tour 1
  RandomForest : 2 tour(s), convergé : le réglage choisi ne change plus ; version retenue : tour 1
  CatBoost : 3 tour(s), convergé : le réglage choisi ne change plus ; version retenue : tour 2
Variante « Sans les PAY_n »
  LogisticRegression : 1 tour(s), optimisé : ni surapprentissage ni paramètre au bord ; version retenue 

**PAY_1 ≥ 2 : détection en validation, au seuil de la règle marginale** (précision marginale visée : 10% de défauts parmi les derniers signalés ; 4.0 min)

Réglage retenu  Écart relatif (%) Précision moyenne        ROC AUC   Rappel (%) Précision (%) Précision marginale (%) Clients signalés (%)  Plis sans tri
Variante       Modèle                                                                                                                                                                                                                           
Complet        LogisticRegression                                                               C 0.1                3.8     0.766 ± 0.016  0.600 ± 0.023  100.0 ± 0.0    69.6 ± 0.1              52.4 ± 4.8           99.9 ± 0.1              0
               MLPClassifier                                 alpha 0.01 ; hidden_layer_sizes (64, 32)                6.3     0.763 ± 0.028  0.589 ± 0.032   99.8 ± 0.3    69.6 ± 0.1              55.9 ± 5.6           99.7 ± 0.3              0
               RandomForest                     max_depth 8 ; min_samples_leaf 100 ; n_estimators 300                5.6     0.773 ± 0.013  0.604 ± 0.008   99.9 ± 0.2    69.8 ± 0.3              55.4 ± 3.5           99.5 ± 0.5              0
               CatBoost              depth 2 ; iterations 445 ; l2_leaf_reg 22.5 ; learning_rate 0.02                6.6     0.779 ± 0.010  0.614 ± 0.020  100.0 ± 0.0    69.6 ± 0.1              51.1 ± 8.8          100.0 ± 0.1              0
PAY_n seuls    LogisticRegression                                                               C 0.1                1.8     0.761 ± 0.017  0.591 ± 0.025  100.0 ± 0.0    69.6 ± 0.1              54.6 ± 7.1          100.0 ± 0.0              0
               MLPClassifier                                  alpha 0.1 ; hidden_layer_sizes (64, 32)                1.1     0.755 ± 0.027  0.587 ± 0.041   99.4 ± 0.7    69.8 ± 0.2              57.6 ± 5.9           99.0 ± 1.0              0
               RandomForest                      max_depth 8 ; min_samples_leaf 20 ; n_estimators 300                2.3     0.760 ± 0.016  0.591 ± 0.024  100.0 ± 0.0    69.6 ± 0.1              55.6 ± 6.7          100.0 ± 0.0              0
               CatBoost            depth 2 ; iterations 445 ; l2_leaf_reg 33.8 ; learning_rate 0.0133                1.8     0.762 ± 0.016  0.594 ± 0.029  100.0 ± 0.0    69.6 ± 0.1              55.9 ± 8.5          100.0 ± 0.0              0
Sans les PAY_n LogisticRegression                                                               C 0.1                2.7     0.729 ± 0.008  0.568 ± 0.015  100.0 ± 0.1    69.6 ± 0.1              52.4 ± 3.6           99.9 ± 0.1              0
               MLPClassifier                                  alpha 0.1 ; hidden_layer_sizes (64, 32)                3.7     0.724 ± 0.023  0.536 ± 0.042   99.8 ± 0.4    69.6 ± 0.1              59.2 ± 8.9           99.8 ± 0.4              0
               RandomForest                     max_depth 2 ; min_samples_leaf 100 ; n_estimators 300                6.0     0.746 ± 0.007  0.569 ± 0.005  100.0 ± 0.0    69.6 ± 0.1              53.0 ± 3.8          100.0 ± 0.1              0
               CatBoost            depth 2 ; iterations 297 ; l2_leaf_reg 33.8 ; learning_rate 0.0133                6.6     0.758 ± 0.005  0.586 ± 0.008  100.0 ± 0.1    69.6 ± 0.1              50.8 ± 6.2           99.9 ± 0.1              0

### PAY_1 < 2 : 26870 clients, taux de défaut 16.6 % ; tolérance 1 (signalé en défaut dès 50% de risque à la marge)

Variante « Complet »
  LogisticRegression : 1 tour(s), optimisé : ni surapprentissage ni paramètre au bord ; version retenue : tour 1
  MLPClassifier : 2 tour(s), convergé : le réglage choisi ne change plus ; version retenue : tour 1
  RandomForest : 2 tour(s), convergé : le réglage choisi ne change plus ; version retenue : tour 1
  CatBoost : 3 tour(s), convergé : le réglage choisi ne change plus ; version retenue : tour 2
Variante « PAY_n seuls »
  LogisticRegression : 1 tour(s), optimisé : ni surapprentissage ni paramètre au bord ; version retenue : tour 1
  MLPClassifier : 1 tour(s), optimisé : ni surapprentissage ni paramètre au bord ; version retenue : tour 1
  RandomForest : 2 tour(s), convergé : le réglage choisi ne change plus ; version retenue : tour 1
  CatBoost : 3 tour(s), convergé : le réglage choisi ne change plus ; version retenue : tour 2
Variante « Sans les PAY_n »
  LogisticRegression : 2 tour(s), convergé : le réglage choisi ne change plus ; version retenue : tour 1

**PAY_1 < 2 : détection en validation, au seuil de la règle marginale** (précision marginale visée : 50% de défauts parmi les derniers signalés ; 24.2 min)

Réglage retenu  Écart relatif (%) Précision moyenne        ROC AUC Rappel (%) Précision (%) Précision marginale (%) Clients signalés (%)  Plis sans tri
Variante       Modèle                                                                                                                                                                                                                          
Complet        LogisticRegression                                                                C 0.1                1.7     0.339 ± 0.011  0.704 ± 0.005  6.5 ± 2.3    52.0 ± 5.0              55.4 ± 8.0            2.1 ± 0.8              0
               MLPClassifier                                   alpha 1.0 ; hidden_layer_sizes (64, 32)                1.9     0.339 ± 0.009  0.705 ± 0.006  1.9 ± 2.9   52.1 ± 31.8             60.4 ± 37.0            0.6 ± 0.8              0
               RandomForest                      max_depth 6 ; min_samples_leaf 100 ; n_estimators 300                5.7     0.337 ± 0.015  0.705 ± 0.006  4.3 ± 2.0    51.1 ± 5.4              55.0 ± 7.9            1.5 ± 0.8              0
               CatBoost            depth 3 ; iterations 1000 ; l2_leaf_reg 22.5 ; learning_rate 0.0133                5.0     0.349 ± 0.013  0.716 ± 0.007  5.2 ± 3.6   39.3 ± 20.0             37.7 ± 12.5            1.8 ± 1.2              0
PAY_n seuls    LogisticRegression                                                                C 0.1                0.6     0.312 ± 0.010  0.662 ± 0.008  5.0 ± 0.6    53.2 ± 4.3              48.7 ± 8.8            1.6 ± 0.2              0
               MLPClassifier                                   alpha 0.1 ; hidden_layer_sizes (32, 16)                1.8     0.312 ± 0.010  0.660 ± 0.007  5.7 ± 2.5    55.0 ± 5.3             47.6 ± 13.7            1.8 ± 0.9              0
               RandomForest                       max_depth 8 ; min_samples_leaf 20 ; n_estimators 200                2.5     0.309 ± 0.012  0.661 ± 0.007  4.9 ± 0.8    52.7 ± 6.1             53.5 ± 24.5            1.5 ± 0.2              0
               CatBoost               depth 2 ; iterations 667 ; l2_leaf_reg 22.5 ; learning_rate 0.03                2.4     0.313 ± 0.010  0.662 ± 0.007  4.0 ± 1.9    51.8 ± 3.7             34.5 ± 20.9            1.3 ± 0.6              0
Sans les PAY_n LogisticRegression                                                               C 10.0                0.5     0.316 ± 0.012  0.688 ± 0.007  4.7 ± 1.0    51.0 ± 3.1              52.0 ± 6.0            1.5 ± 0.3              0
               MLPClassifier                                     alpha 0.01 ; hidden_layer_sizes (32,)                3.7     0.319 ± 0.007  0.691 ± 0.004  2.0 ± 1.8    55.3 ± 8.6             64.0 ± 19.6            0.7 ± 0.7              0
               RandomForest                      max_depth 4 ; min_samples_leaf 100 ; n_estimators 300                5.9     0.298 ± 0.008  0.680 ± 0.008  0.3 ± 0.3   40.3 ± 38.8             66.7 ± 47.1            0.1 ± 0.1              0
               CatBoost                depth 2 ; iterations 1000 ; l2_leaf_reg 15 ; learning_rate 0.03                6.9     0.333 ± 0.012  0.707 ± 0.007  2.5 ± 2.8   53.0 ± 11.7             52.2 ± 33.9            0.9 ± 1.1              0

**Comment lire** : chaque population a son propre taux de défaut, qui est aussi la précision d'un tri au hasard. Côté `PAY_1` ≥ 2, la question est de reconnaître ceux qui **ne feront pas** défaut : avec la même tolérance stricte que pour la population en incident et le retard imminent (9 chances sur 10 d'être sain), seuls les clients au risque très bas sont considérés comme sains ; si la part des clients signalés reste proche de 100 %, le modèle ne sait pas trancher. Côté autres clients, la règle ne signale que la tête de liste. La précision marginale vérifie la règle (proche de 10 % côté `PAY_1` ≥ 2, de 50 % côté autres clients).

## 4. Ce qui porte les prédictions, dans chaque population

Importance par permutation (perte de précision moyenne quand une famille de colonnes est mélangée, sur les plis de validation), comme dans `essai_jeu_corrige.ipynb` : part de chaque famille dans la perte totale, par population, variante et modèle.

In [7]:
FAMILLES = {
    "Codifications (PAY_n)": PAY,
    "Factures (BILL_AMTn)": [f"BILL_AMT{n}" for n in range(1, 7)],
    "Paiements (PAY_AMTn)": [f"PAY_AMT{n}" for n in range(1, 7)],
    "Plafond (LIMIT_BAL)": ["LIMIT_BAL"],
    "Démographie": ["SEX", "EDUCATION", "MARRIAGE", "AGE"],
}
REPETITIONS = 5
generateur = np.random.default_rng(42)

def perte_par_permutation(modeles, X, colonnes):
    # Perte de précision moyenne sur chaque pli de validation quand les colonnes sont mélangées d'un bloc (même tirage pour toutes),
    # moyennée sur les répétitions
    pertes = []
    for m, (_, validation) in zip(modeles, plis):
        X_val, y_val = X.iloc[validation], y[validation]
        reference = average_precision_score(y_val, m.predict_proba(X_val)[:, 1])
        perte = []
        for _ in range(REPETITIONS):
            melange = X_val.copy()
            ordre_melange = generateur.permutation(len(melange))
            melange[colonnes] = melange[colonnes].to_numpy()[ordre_melange]
            perte.append(reference - average_precision_score(y_val, m.predict_proba(melange)[:, 1]))
        pertes.append(np.mean(perte))
    return pertes


for population, e in ETUDE.items():
    activer(population)
    pertes = {}
    for (variante, nom), modeles in e["modeles_plis"].items():
        X = original[COLONNES_VARIANTE[variante]].reset_index(drop=True)
        for famille, colonnes in FAMILLES.items():
            presentes = [c for c in colonnes if c in X.columns]
            if presentes:
                pertes[(variante, nom, famille)] = np.mean(perte_par_permutation(modeles, X, presentes))
    cles = list(dict.fromkeys((v, n) for v, n, _ in pertes))
    tableau = pd.DataFrame({cle: {f: pertes[(*cle, f)] for f in FAMILLES if (*cle, f) in pertes} for cle in cles})
    part = tableau.clip(lower=0) / tableau.clip(lower=0).sum() * 100
    display(Markdown(f"**{population} : part de chaque famille dans la perte de précision moyenne (%)**"))
    display(part.round(1).rename_axis(index="Famille", columns=["Variante", "Modèle"]))

**PAY_1 ≥ 2 : part de chaque famille dans la perte de précision moyenne (%)**

Variante                         Complet                                            PAY_n seuls                                         Sans les PAY_n                                    
Modèle                LogisticRegression MLPClassifier RandomForest CatBoost LogisticRegression MLPClassifier RandomForest CatBoost LogisticRegression MLPClassifier RandomForest CatBoost
Famille                                                                                                                                                                                   
Codifications (PAY_n)               61.6          55.6         55.2     56.2              100.0         100.0        100.0    100.0                NaN           NaN          NaN      NaN
Factures (BILL_AMTn)                 1.4          15.5         27.9     18.4                NaN           NaN          NaN      NaN               33.2          50.7         59.1     51.6
Paiements (PAY_AMTn)                21.8          21.4          9.0     13.4                NaN           NaN          NaN      NaN               19.6          36.2         22.1     24.6
Plafond (LIMIT_BAL)                  0.0           0.0          1.8      3.4                NaN           NaN          NaN      NaN                0.0          13.0          0.0      4.6
Démographie                         15.2           7.5          6.2      8.6                NaN           NaN          NaN      NaN               47.2           0.0         18.8     19.2

**PAY_1 < 2 : part de chaque famille dans la perte de précision moyenne (%)**

Variante                         Complet                                            PAY_n seuls                                         Sans les PAY_n                                    
Modèle                LogisticRegression MLPClassifier RandomForest CatBoost LogisticRegression MLPClassifier RandomForest CatBoost LogisticRegression MLPClassifier RandomForest CatBoost
Famille                                                                                                                                                                                   
Codifications (PAY_n)               35.9          32.2         69.9     53.7              100.0         100.0        100.0    100.0                NaN           NaN          NaN      NaN
Factures (BILL_AMTn)                 8.6          11.2          2.6      9.3                NaN           NaN          NaN      NaN               24.7          24.7         16.2     28.6
Paiements (PAY_AMTn)                45.2          45.9         21.9     23.1                NaN           NaN          NaN      NaN               61.9          63.7         67.0     56.8
Plafond (LIMIT_BAL)                  7.0           7.9          4.5     11.0                NaN           NaN          NaN      NaN                9.9           7.1         14.6     12.0
Démographie                          3.3           2.7          1.2      2.9                NaN           NaN          NaN      NaN                3.5           4.4          2.2      2.6

## 5. Synthèse globale : résultats accumulés sur tous les clients du jeu de base

Chaque client reçoit la décision (et la probabilité) de sa population, hors pli. Le **meilleur modèle** de chaque population est celui de la variante choisie qui a la plus haute précision moyenne hors pli (règle fixée avant de lancer). Les mesures portent sur **tous les clients** : défauts détectés (nombre et part de tous les défauts), clients signalés, précision des défauts prédits, fausses alertes, défauts manqués, taux d'erreur (fausses alertes et défauts manqués, sur tous les clients), et ratio d'aire (Yeh et Lien : aire entre la courbe de gain et la diagonale, rapportée à celle de la courbe parfaite ; il vaut 2 × ROC AUC − 1).

Pour le ratio d'aire, les clients sont triés par une note : la règle de la banque place les clients `PAY_1` ≥ 2 en tête ; « deux ML » trie tous les clients par la probabilité de leur modèle (deux modèles différents : probabilités pas tout à fait comparables, ratio indicatif).

In [8]:
from sklearn.metrics import roc_auc_score

def meilleur_modele(population, variante):
    e = ETUDE[population]
    return max([n for v, n in e["probas"] if v == variante], key=lambda n: average_precision_score(POPULATIONS[population]["dpnm"], e["probas"][(variante, n)]))

def vecteurs(population, variante):
    # Probabilités et décisions hors pli du meilleur modèle d'une population, rangées par identifiant client
    nom = meilleur_modele(population, variante)
    ids = POPULATIONS[population]["ID"].to_numpy()
    return nom, pd.Series(ETUDE[population]["probas"][(variante, nom)], index=ids), pd.Series(ETUDE[population]["decisions"][(variante, nom)], index=ids)

ids_tous = donnees["ID"].to_numpy()
y_tous = donnees["dpnm"].to_numpy()
ctx = pd.Series(au_ctx, index=ids_tous)

nom_ctx, p_ctx, d_ctx = vecteurs("PAY_1 ≥ 2", "Complet")
nom_hors, p_hors, d_hors = vecteurs("PAY_1 < 2", "Complet")
nom_hors_sp, p_hors_sp, d_hors_sp = vecteurs("PAY_1 < 2", "Sans les PAY_n")

def assembler(dec_ctx, dec_hors, note_ctx, note_hors):
    dec = pd.concat([dec_ctx, dec_hors]).reindex(ids_tous).to_numpy()
    note = pd.concat([note_ctx, note_hors]).reindex(ids_tous).to_numpy()
    return dec.astype(int), note

ids_ctx, ids_hors = POPULATIONS["PAY_1 ≥ 2"]["ID"], POPULATIONS["PAY_1 < 2"]["ID"]
un = lambda ids: pd.Series(1, index=ids)
zero = lambda ids: pd.Series(0, index=ids)
STRATEGIES = {
    "1. Règle de la banque seule (PAY_1 ≥ 2 en défaut)": assembler(un(ids_ctx), zero(ids_hors), un(ids_ctx) * 2.0, zero(ids_hors) * 1.0),
    f"2. Règle de la banque + ML pour les autres ({nom_hors})": assembler(un(ids_ctx), d_hors, un(ids_ctx) * 2.0, p_hors),
    f"3. Règle de la banque + ML pour les autres sans les PAY_n ({nom_hors_sp})": assembler(un(ids_ctx), d_hors_sp, un(ids_ctx) * 2.0, p_hors_sp),
    f"4. Deux ML ({nom_ctx} pour PAY_1 ≥ 2, {nom_hors} pour les autres)": assembler(d_ctx, d_hors, p_ctx, p_hors),
}
# 5. Un seul ML sur tous les clients : décisions enregistrées par essai_jeu_de_base.ipynb (cellule d'enregistrement)
fichier_un_ml = root_dir / "data" / "ML" / "comparatif_global" / "jeu_de_base.csv"
if fichier_un_ml.exists():
    un_ml = pd.read_csv(fichier_un_ml).set_index("ID").loc[ids_tous]
    noms = sorted({c.split(" | ")[2] for c in un_ml.columns if c.startswith("proba | Complet |")})
    nom_un = max(noms, key=lambda n: average_precision_score(y_tous, un_ml[f"proba | Complet | {n}"]))
    STRATEGIES[f"5. Un seul ML sur tous les clients ({nom_un})"] = (un_ml[f"decision | Complet | {nom_un}"].to_numpy().astype(int), un_ml[f"proba | Complet | {nom_un}"].to_numpy())
else:
    display(Markdown("⚠️ Stratégie 5 absente : lancer `essai_jeu_de_base.ipynb` avec sa cellule d'enregistrement (fin de la section 4)."))

def ratio_aire(y_vrai, note):
    n, d = len(y_vrai), y_vrai.sum()
    cumul = np.cumsum(y_vrai[np.argsort(-note, kind="mergesort")])
    diagonale = d * (n + 1) / 2
    return (cumul.sum() - diagonale) / (np.minimum(np.arange(1, n + 1), d).sum() - diagonale)

lignes = []
for nom, (dec, note) in STRATEGIES.items():
    signale = dec == 1
    detectes = (signale & (y_tous == 1)).sum()
    fausses = (signale & (y_tous == 0)).sum()
    manques = (~signale & (y_tous == 1)).sum()
    lignes.append({"Stratégie": nom, "Défauts détectés": detectes, "Part des défauts (%)": round(detectes / y_tous.sum() * 100, 1),
                   "Clients signalés": signale.sum(), "Part des clients signalés (%)": round(signale.mean() * 100, 1),
                   "Précision des défauts prédits (%)": round(detectes / signale.sum() * 100, 1) if signale.sum() else np.nan,
                   "Fausses alertes": fausses, "Défauts manqués": manques,
                   "Taux d'erreur": round((fausses + manques) / len(y_tous), 3), "Ratio d'aire": round(ratio_aire(y_tous, note), 3)})
lignes.append({"Stratégie": "Personne n'est signalé (repère)", "Défauts détectés": 0, "Part des défauts (%)": 0.0, "Clients signalés": 0,
               "Part des clients signalés (%)": 0.0, "Précision des défauts prédits (%)": np.nan, "Fausses alertes": 0,
               "Défauts manqués": y_tous.sum(), "Taux d'erreur": round(y_tous.mean(), 3), "Ratio d'aire": 0.0})
display(Markdown(f"**Résultats accumulés sur tous les clients du jeu de base** ({len(y_tous)} clients, {y_tous.sum()} défauts, taux de défaut {y_tous.mean() * 100:.1f} %)"))
display(pd.DataFrame(lignes).set_index("Stratégie"))

# Détail par population : d'où viennent les défauts détectés et les fausses alertes
detail = []
for nom, (dec, _) in STRATEGIES.items():
    for population, masque in (("PAY_1 ≥ 2", au_ctx), ("PAY_1 < 2", ~au_ctx)):
        s, yy = dec[masque] == 1, y_tous[masque]
        detail.append({"Stratégie": nom, "Population": population, "Défauts détectés": (s & (yy == 1)).sum(), "sur": yy.sum(),
                       "Fausses alertes": (s & (yy == 0)).sum(), "Précision (%)": round((s & (yy == 1)).sum() / s.sum() * 100, 1) if s.sum() else np.nan})
display(Markdown("**Détail par population**"))
display(pd.DataFrame(detail).set_index(["Stratégie", "Population"]))

# Comparatif avec l'étude d'origine : ses 6 méthodes (Tableau 1, en validation) et nos stratégies globales, sur les deux mesures qu'elle publie
YEH_LIEN = {   # méthode : (taux d'erreur, ratio d'aire), en validation
    "Réseau de neurones": (0.17, 0.54), "Arbre de classification": (0.17, 0.536), "Bayésien naïf": (0.21, 0.53),
    "KNN": (0.16, 0.45), "Régression logistique": (0.18, 0.44), "Analyse discriminante": (0.26, 0.43),
}
comparatif = [{"Source": "Yeh et Lien (2009)", "Méthode ou stratégie": m, "Taux d'erreur": e, "Ratio d'aire": a,
               "Périmètre": "fichier brut, tous les clients", "Découpage": "un seul (entraînement / validation)"} for m, (e, a) in YEH_LIEN.items()]
comparatif += [{"Source": "Cet essai", "Méthode ou stratégie": l["Stratégie"], "Taux d'erreur": l["Taux d'erreur"], "Ratio d'aire": l["Ratio d'aire"],
                "Périmètre": f"fichier brut, niveau 0 ({len(y_tous)} clients) : même périmètre", "Découpage": "validation croisée à 5 plis (décisions hors pli)"} for l in lignes]
display(Markdown("**Comparatif avec l'étude d'origine (Yeh et Lien, 2009, Tableau 1, en validation)** : taux d'erreur et ratio d'aire, "
                 "les deux seules mesures qu'elle publie (ni rappel ni précision)"))
display(pd.DataFrame(comparatif).set_index(["Source", "Méthode ou stratégie"]))
display(Markdown(
    "**Même périmètre que l'étude d'origine** : le fichier brut, tous les clients. Restent deux différences : le découpage (une validation croisée "
    "à 5 plis ici, un seul découpage entraînement / validation dans l'étude) et les arrondis de l'étude, au centième."))

# Décisions client par client, pour l'accumulation ultérieure
sortie = pd.DataFrame({"ID": ids_tous, "dpnm": y_tous, "pay1_retard": au_ctx.astype(int)})
for nom, (dec, note) in STRATEGIES.items():
    sortie[f"decision | {nom}"] = dec
    sortie[f"note | {nom}"] = note
dossier_sortie = root_dir / "data" / "ML" / "comparatif_global"
dossier_sortie.mkdir(parents=True, exist_ok=True)
sortie.to_csv(dossier_sortie / "jeu_de_base_deux_populations.csv", index=False)
print(f"Décisions enregistrées : {dossier_sortie / 'jeu_de_base_deux_populations.csv'}")

**Résultats accumulés sur tous les clients du jeu de base** (30000 clients, 6636 défauts, taux de défaut 22.1 %)

,Défauts détectés,Part des défauts (%),Clients signalés,Part des clients signalés (%),Précision des défauts prédits (%),Fausses alertes,Défauts manqués,Taux d'erreur,Ratio d'aire
Stratégie,,,,,,,,,
1. Règle de la banque seule (PAY_1 ≥ 2 en défaut),2177,32.8,3130,10.4,69.6,953,4459,0.180,0.308
2. Règle de la banque + ML pour les autres (CatBoost),2411,36.3,3606,12.0,66.9,1195,4225,0.181,0.564
3. Règle de la banque + ML pour les autres sans les PAY_n (CatBoost),2289,34.5,3376,11.3,67.8,1087,4347,0.181,0.553
"4. Deux ML (CatBoost pour PAY_1 ≥ 2, CatBoost pour les autres)",2411,36.3,3605,12.0,66.9,1194,4225,0.181,0.423
5. Un seul ML sur tous les clients (MLPClassifier),2430,36.6,3646,12.2,66.6,1216,4206,0.181,0.558
Personne n'est signalé (repère),0,0.0,0,0.0,NaN,0,6636,0.221,0.000


**Détail par population**

Défauts détectés   sur  Fausses alertes  Précision (%)
Stratégie                                                            Population                                                        
1. Règle de la banque seule (PAY_1 ≥ 2 en défaut)                    PAY_1 ≥ 2               2177  2177              953           69.6
                                                                     PAY_1 < 2                  0  4459                0            NaN
2. Règle de la banque + ML pour les autres (CatBoost)                PAY_1 ≥ 2               2177  2177              953           69.6
                                                                     PAY_1 < 2                234  4459              242           49.2
3. Règle de la banque + ML pour les autres sans les PAY_n (CatBoost) PAY_1 ≥ 2               2177  2177              953           69.6
                                                                     PAY_1 < 2                112  4459              134           45.5
4. Deux ML (CatBoost pour PAY_1 ≥ 2, CatBoost pour les autres)       PAY_1 ≥ 2               2177  2177              952           69.6
                                                                     PAY_1 < 2                234  4459              242           49.2
5. Un seul ML sur tous les clients (MLPClassifier)                   PAY_1 ≥ 2               2086  2177              864           70.7
                                                                     PAY_1 < 2                344  4459              352           49.4

**Comparatif avec l'étude d'origine (Yeh et Lien, 2009, Tableau 1, en validation)** : taux d'erreur et ratio d'aire, les deux seules mesures qu'elle publie (ni rappel ni précision)

Taux d'erreur  Ratio d'aire                                                Périmètre                                         Découpage
Source             Méthode ou stratégie                                                                                                                                                                                        
Yeh et Lien (2009) Réseau de neurones                                                            0.170         0.540                           fichier brut, tous les clients               un seul (entraînement / validation)
                   Arbre de classification                                                       0.170         0.536                           fichier brut, tous les clients               un seul (entraînement / validation)
                   Bayésien naïf                                                                 0.210         0.530                           fichier brut, tous les clients               un seul (entraînement / validation)
                   KNN                                                                           0.160         0.450                           fichier brut, tous les clients               un seul (entraînement / validation)
                   Régression logistique                                                         0.180         0.440                           fichier brut, tous les clients               un seul (entraînement / validation)
                   Analyse discriminante                                                         0.260         0.430                           fichier brut, tous les clients               un seul (entraînement / validation)
Cet essai          1. Règle de la banque seule (PAY_1 ≥ 2 en défaut)                             0.180         0.308  fichier brut, niveau 0 (30000 clients) : même périmètre  validation croisée à 5 plis (décisions hors pli)
                   2. Règle de la banque + ML pour les autres (CatBoost)                         0.181         0.564  fichier brut, niveau 0 (30000 clients) : même périmètre  validation croisée à 5 plis (décisions hors pli)
                   3. Règle de la banque + ML pour les autres sans les PAY_n (CatBoost)          0.181         0.553  fichier brut, niveau 0 (30000 clients) : même périmètre  validation croisée à 5 plis (décisions hors pli)
                   4. Deux ML (CatBoost pour PAY_1 ≥ 2, CatBoost pour les autres)                0.181         0.423  fichier brut, niveau 0 (30000 clients) : même périmètre  validation croisée à 5 plis (décisions hors pli)
                   5. Un seul ML sur tous les clients (MLPClassifier)                            0.181         0.558  fichier brut, niveau 0 (30000 clients) : même périmètre  validation croisée à 5 plis (décisions hors pli)
                   Personne n'est signalé (repère)                                               0.221         0.000  fichier brut, niveau 0 (30000 clients) : même périmètre  validation croisée à 5 plis (décisions hors pli)

**Même périmètre que l'étude d'origine** : le fichier brut, tous les clients. Restent deux différences : le découpage (une validation croisée à 5 plis ici, un seul découpage entraînement / validation dans l'étude) et les arrondis de l'étude, au centième.

Décisions enregistrées : c:\Users\johan\VS Code Wild 2\PROJET_DEFAULT_CREDIT_CARD_ML\data\ML\comparatif_global\jeu_de_base_deux_populations.csv


**Comment lire** :
- **Défauts détectés et précision** : combien de défauts chaque stratégie récupère en tout, et à quel prix (fausses alertes). La règle de la banque seule donne le repère : les clients déjà en retard en septembre, sans modèle.
- **Règle + ML** (2 et 3) : ce que le modèle ajoute aux clients en retard, en ne signalant parmi les autres que la tête de liste (règle marginale). La stratégie 3 montre ce que le **comportement seul** (les montants, sans codification) ajoute.
- **Deux ML et un seul ML** (4 et 5) : si un modèle fait mieux que la règle sur les clients en retard, et si séparer les populations change quelque chose.
- **Comparatif avec l'étude d'origine** : même périmètre ; restent le découpage et les arrondis de l'étude.

## 6. Comparaison avec la séparation par la règle métier, sur les clients communs

Les décisions de ce notebook (codification de la banque, jeu de base) face à celles de `essai_jeu_corrige_deux_populations.ipynb` (règle métier : population en incident à M et retard imminent, jeu corrigé), **sur les clients communs aux deux** : ceux du niveau 3. Les clients retirés au nettoyage sont écartés de la comparaison (les modèles du jeu de base ont cependant appris aussi sur eux). Mêmes mesures qu'en section 5, recalculées sur ces seuls clients. Le premier tableau croise les deux séparations : combien de clients en incident à M ou en retard imminent sont aussi en retard en septembre selon la banque, et l'inverse.

In [9]:
fichier_ctx = root_dir / "data" / "ML" / "comparatif_global" / "jeu_corrige_deux_populations.csv"
if not fichier_ctx.exists():
    display(Markdown("⚠️ Comparaison impossible : lancer d'abord `essai_jeu_corrige_deux_populations.ipynb` (il enregistre ses décisions)."))
else:
    travail = pd.read_csv(fichier_ctx).set_index("ID")
    banque = sortie.set_index("ID")
    communs = travail.index.intersection(banque.index)
    assert (travail.loc[communs, "dpnm"] == banque.loc[communs, "dpnm"]).all(), "Cible différente entre les deux fichiers"
    y_c = banque.loc[communs, "dpnm"].to_numpy()
    display(Markdown(f"**Clients communs** : {len(communs)} (sur {len(banque)} du jeu de base et {len(travail)} du jeu corrigé), dont {y_c.sum()} défauts"))
    croise = pd.crosstab(travail.loc[communs, "population"].map({"En incident": "en incident à M", "Retard imminent": "retard imminent", "Autres clients": "autres clients"}),
                         banque.loc[communs, "pay1_retard"].map({1: "PAY_1 ≥ 2 (banque)", 0: "PAY_1 < 2"}), margins=True, margins_name="Total")
    display(Markdown("**Les deux séparations, croisées**"))
    display(croise)

    def mesures(dec, note):
        signale = dec == 1
        detectes, fausses, manques = (signale & (y_c == 1)).sum(), (signale & (y_c == 0)).sum(), (~signale & (y_c == 1)).sum()
        return {"Défauts détectés": detectes, "Part des défauts (%)": round(detectes / y_c.sum() * 100, 1), "Clients signalés": signale.sum(),
                "Précision des défauts prédits (%)": round(detectes / signale.sum() * 100, 1) if signale.sum() else np.nan,
                "Fausses alertes": fausses, "Défauts manqués": manques, "Taux d'erreur": round((fausses + manques) / len(y_c), 3),
                "Ratio d'aire": round(ratio_aire(y_c, note), 3)}

    lignes_c = []
    for source, table in (("Codification de la banque (PAY_1 ≥ 2), jeu de base", banque), ("Règle métier (en incident à M, retard imminent), jeu corrigé", travail)):
        for col in [c for c in table.columns if c.startswith("decision | ")]:
            strategie = col.removeprefix("decision | ")
            lignes_c.append({"Séparation": source, "Stratégie": strategie,
                             **mesures(table.loc[communs, col].to_numpy(), table.loc[communs, f"note | {strategie}"].to_numpy())})
    display(Markdown("**Les deux séparations face à face, sur les mêmes clients**"))
    display(pd.DataFrame(lignes_c).set_index(["Séparation", "Stratégie"]))

**Clients communs** : 28851 (sur 30000 du jeu de base et 28851 du jeu corrigé), dont 6272 défauts

**Les deux séparations, croisées**

pay1_retard,PAY_1 < 2,PAY_1 ≥ 2 (banque),Total
population,,,
autres clients,24108,0,24108
en incident à M,1623,2131,3754
retard imminent,0,989,989
Total,25731,3120,28851


**Les deux séparations face à face, sur les mêmes clients**

Défauts détectés  Part des défauts (%)  Clients signalés  Précision des défauts prédits (%)  \
Séparation                                                   Stratégie                                                                                                                                                                                 
Codification de la banque (PAY_1 ≥ 2), jeu de base           1. Règle de la banque seule (PAY_1 ≥ 2 en défaut)                                                       2169                  34.6              3120                               69.5   
                                                             2. Règle de la banque + ML pour les autres (CatBoost)                                                   2403                  38.3              3596                               66.8   
                                                             3. Règle de la banque + ML pour les autres sans les PAY_n (CatBoost)                                    2279                  36.3              3363                               67.8   
                                                             4. Deux ML (CatBoost pour PAY_1 ≥ 2, CatBoost pour les autres)                                          2403                  38.3              3595                               66.8   
                                                             5. Un seul ML sur tous les clients (MLPClassifier)                                                      2418                  38.6              3626                               66.7   
Règle métier (en incident à M, retard imminent), jeu corrigé 1. Règle seule (en incident et retard imminent en défaut)                                               2871                  45.8              4743                               60.5   
                                                             2. Règle + ML autres clients (CatBoost)                                                                 2871                  45.8              4745                               60.5   
                                                             3. Règle + ML autres clients sans les PAY_n (CatBoost)                                                  2877                  45.9              4753                               60.5   
                                                             4. Trois ML (CatBoost en incident, MLPClassifier retard imminent, CatBoost autres clients)              2868                  45.7              4738                               60.5   
                                                             5. Un seul ML sur tous les clients (CatBoost)                                                           2527                  40.3              4010                               63.0   

                                                                                                                                                         Fausses alertes  Défauts manqués  Taux d'erreur  Ratio d'aire  
Séparation                                                   Stratégie                                                                                                                                                  
Codification de la banque (PAY_1 ≥ 2), jeu de base           1. Règle de la banque seule (PAY_1 ≥ 2 en défaut)                                                       951             4103          0.175         0.324  
                                                             2. Règle de la banque + ML pour les autres (CatBoost)                                                  1193             3869          0.175         0.565  
                                                             3. Règle de la banque + ML pour les autres sans les PAY_n (CatBoost)                                   1084             3993          0.176         0.554  
                                                             4. Deux ML (CatBoost pour PAY_

**Comment lire** : à stratégie équivalente (règle seule, règle + ML, deux ML, un seul ML), une séparation fait mieux si elle détecte plus de défauts pour une précision au moins égale, ou la même part de défauts avec moins de fausses alertes. Le tableau croisé dit d'où viennent les écarts : clients en incident à M ou en retard imminent que la banque ne voit pas en retard en septembre, ou clients en retard en septembre qui ne relèvent pas de la règle métier (retards isolés, codifications non corrigées). La comparaison mêle la règle de séparation et le nettoyage (jeu de base contre jeu corrigé) : elle répond à la question « tout le travail fait-il mieux que la codification de la banque », pas à « la règle seule fait-elle mieux ».

## 7. Conclusion : la règle métier face à la codification de la banque

*Lancement du 12/10/2026 : niveau 0, encodage des scénarios actuels (codifications en catégories, plafond au logarithme), comparaison avec la séparation par la règle métier (population en incident à M, retard imminent).*

**1. Le groupe `PAY_1` ≥ 2 se comporte comme les populations de la règle métier** (section 3) : avec la même exigence (9 chances sur 10 d'être sain pour en sortir), les modèles gardent la quasi-totalité du groupe en défaut, et y trient à peine mieux que le hasard (ROC AUC de 0,54 à 0,61). Le peu qu'ils trouvent passe d'abord par les codifications, puis par les factures et les paiements (section 4).

**2. Pour les autres clients (`PAY_1` < 2), la règle « 1 pour 1 » ne signale qu'une petite tête de liste** : environ 2 % des clients au plus, avec une précision proche de 50 % (section 3). Le modèle les classe un peu mieux que les autres clients du jeu corrigé (ROC AUC d'environ 0,70 dans la variante complète) : ce groupe contient encore des clients que la règle métier place en incident (point 4).

**3. Résultats accumulés sur le même périmètre que l'étude d'origine** (section 5) : la codification de la banque seule trouve environ un tiers des défauts, avec environ sept prédictions justes sur dix, en signalant environ un client sur dix ; le modèle pour les autres clients en ajoute un peu, à environ un défaut pour un bon client signalé. Le taux d'erreur ne bouge pas d'une stratégie à l'autre ; la qualité du tri (ratio d'aire) monte nettement dès qu'un modèle range les autres clients. Face à Yeh et Lien, sur le même fichier : ratio d'aire au niveau de leur meilleure méthode ou un peu au-dessus, taux d'erreur au niveau de leur régression logistique.

**4. Les deux séparations, sur les mêmes clients** (section 6) :
- **elles s'emboîtent** : tous les clients que la banque codifie en retard en septembre sont aussi dans la règle métier ; la règle métier y ajoute des clients en incident à M que la banque ne codifie pas en retard en septembre : tous codifiés 1 par la banque, et jugés en retard une fois cette codification corrigée au nettoyage ;
- **la codification de la banque retient moins de clients, avec plus de précision, mais trouve moins de défauts** ; la règle métier en trouve nettement plus, avec une précision plus basse ; les clients qu'elle ajoute font défaut moins d'une fois sur deux ;
- avec le modèle pour les autres clients, la séparation par la banque a un taux d'erreur un peu plus bas et une qualité de tri du même ordre : ce sont deux points de la même courbe (`niveaux_de_risque_global.ipynb`).

**Ce que ça veut dire** : la règle métier ne **prédit pas mieux** que la codification de la banque ; elle va plus loin dans le portefeuille, en retenant aussi des clients en incident que la banque ne codifie pas en retard en septembre, au prix d'une précision plus basse. Sa valeur est d'être **justifiée et explicable** (deux retards d'affilée, ou un retard grave ; codifications relues avec les paiements). La comparaison mêle la règle et le nettoyage (jeu de base contre jeu corrigé) : elle répond à « tout le travail fait-il mieux que la codification de la banque ? », et la réponse, sur le score, est non : ce sont deux points de la même courbe.

**Réserves** : les modèles ont appris sur les clients du test du ML ; « dans le bruit » veut dire qu'on ne peut pas départager, pas que les scores sont identiques. Ces résultats éclairent la problématique, ils ne servent pas à choisir le modèle final.

## 8. Complément : la stratégie 5 (un seul ML), lue dans les fichiers enregistrés

Cette cellule est **autonome** : elle ne réentraîne rien et fonctionne même après une réinitialisation du noyau. Elle lit les décisions enregistrées par ce notebook et par l'essai à un seul modèle (`essai_jeu_de_base.ipynb`, cellule d'enregistrement), ajoute la stratégie 5 (meilleur modèle unique de la variante complète, choisi sur la précision moyenne hors pli) aux décisions enregistrées, puis refait la synthèse globale, le détail par population et le comparatif avec l'étude d'origine. Elle refait aussi la comparaison avec la séparation par la règle métier, stratégie 5 comprise des deux côtés : **lancer d'abord la même cellule dans `essai_jeu_corrige_deux_populations.ipynb`**.

La stratégie 5 garde sa propre règle : un seul modèle, une seule tolérance (1 pour 1) pour tous les clients ; sans séparation, la règle métier n'existe pas, et le modèle répond à la seule question « qui paiera, qui ne paiera pas ? ».

In [10]:
FICHIER_DEUX, FICHIER_UN, NOTEBOOK_UN = "jeu_de_base_deux_populations.csv", "jeu_de_base.csv", "essai_jeu_de_base.ipynb"
COLONNE_REGLE, NOMS_POPULATIONS = "pay1_retard", ("PAY_1 ≥ 2", "PAY_1 < 2")
PERIMETRE = "fichier brut : même périmètre que l'étude"
# Complément : stratégie 5 (un seul ML sur tous les clients), lue dans les fichiers enregistrés ; cellule autonome, ne réentraîne rien
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display
from sklearn.metrics import average_precision_score

root_dir = next(p for p in [Path.cwd()] + list(Path.cwd().parents) if (p / "data").exists())
dossier = root_dir / "data" / "ML" / "comparatif_global"
deux = pd.read_csv(dossier / FICHIER_DEUX).set_index("ID")
assert (dossier / FICHIER_UN).exists(), f"{FICHIER_UN} absent : lancer {NOTEBOOK_UN} avec sa cellule d'enregistrement"
un = pd.read_csv(dossier / FICHIER_UN).set_index("ID").loc[deux.index]
assert (un["dpnm"] == deux["dpnm"]).all(), "Cible différente entre les deux fichiers"

# Meilleur modèle unique de la variante complète : la plus haute précision moyenne hors pli (même règle que pour les autres stratégies)
noms = sorted({c.split(" | ")[2] for c in un.columns if c.startswith("proba | Complet |")})
nom_un = max(noms, key=lambda n: average_precision_score(un["dpnm"], un[f"proba | Complet | {n}"]))
libelle_5 = f"5. Un seul ML sur tous les clients ({nom_un})"
for c in [c for c in deux.columns if c.startswith(("decision | 5.", "note | 5."))]:
    deux = deux.drop(columns=c)
deux[f"decision | {libelle_5}"] = un[f"decision | Complet | {nom_un}"].astype(int)
deux[f"note | {libelle_5}"] = un[f"proba | Complet | {nom_un}"]
deux.reset_index().to_csv(dossier / FICHIER_DEUX, index=False)   # décisions enregistrées, stratégie 5 comprise

def ratio_aire(y_vrai, note):
    n, d = len(y_vrai), y_vrai.sum()
    cumul = np.cumsum(y_vrai[np.argsort(-note, kind="mergesort")])
    diagonale = d * (n + 1) / 2
    return (cumul.sum() - diagonale) / (np.minimum(np.arange(1, n + 1), d).sum() - diagonale)

def mesures(y, dec, note):
    signale = dec == 1
    detectes, fausses, manques = (signale & (y == 1)).sum(), (signale & (y == 0)).sum(), (~signale & (y == 1)).sum()
    return {"Défauts détectés": detectes, "Part des défauts (%)": round(detectes / y.sum() * 100, 1), "Clients signalés": signale.sum(),
            "Part des clients signalés (%)": round(signale.mean() * 100, 1),
            "Précision des défauts prédits (%)": round(detectes / signale.sum() * 100, 1) if signale.sum() else np.nan,
            "Fausses alertes": fausses, "Défauts manqués": manques, "Taux d'erreur": round((fausses + manques) / len(y), 3),
            "Ratio d'aire": round(ratio_aire(y, note), 3)}

y = deux["dpnm"].to_numpy()
strategies = [c.removeprefix("decision | ") for c in deux.columns if c.startswith("decision | ")]
lignes = [{"Stratégie": s, **mesures(y, deux[f"decision | {s}"].to_numpy(), deux[f"note | {s}"].to_numpy())} for s in strategies]
display(Markdown(f"**Résultats accumulés, stratégie 5 comprise** ({len(y)} clients, {y.sum()} défauts, taux de défaut {y.mean() * 100:.1f} %)"))
display(pd.DataFrame(lignes).set_index("Stratégie"))

regle = deux[COLONNE_REGLE].to_numpy() == 1
detail = []
for s in strategies:
    dec = deux[f"decision | {s}"].to_numpy() == 1
    for population, masque in ((NOMS_POPULATIONS[0], regle), (NOMS_POPULATIONS[1], ~regle)):
        ss, yy = dec[masque], y[masque]
        detail.append({"Stratégie": s, "Population": population, "Défauts détectés": (ss & (yy == 1)).sum(), "sur": yy.sum(),
                       "Fausses alertes": (ss & (yy == 0)).sum(), "Précision (%)": round((ss & (yy == 1)).sum() / ss.sum() * 100, 1) if ss.sum() else np.nan})
display(Markdown("**Détail par population, stratégie 5 comprise**"))
display(pd.DataFrame(detail).set_index(["Stratégie", "Population"]))

YEH_LIEN = {"Réseau de neurones": (0.17, 0.54), "Arbre de classification": (0.17, 0.536), "Bayésien naïf": (0.21, 0.53),
            "KNN": (0.16, 0.45), "Régression logistique": (0.18, 0.44), "Analyse discriminante": (0.26, 0.43)}
comparatif = [{"Source": "Yeh et Lien (2009)", "Méthode ou stratégie": m, "Taux d'erreur": e, "Ratio d'aire": a} for m, (e, a) in YEH_LIEN.items()]
comparatif += [{"Source": "Cet essai", "Méthode ou stratégie": l["Stratégie"], "Taux d'erreur": l["Taux d'erreur"], "Ratio d'aire": l["Ratio d'aire"]} for l in lignes]
display(Markdown(f"**Comparatif avec l'étude d'origine, stratégie 5 comprise** ({PERIMETRE})"))
display(pd.DataFrame(comparatif).set_index(["Source", "Méthode ou stratégie"]))

# Comparaison avec la séparation par la règle métier, stratégie 5 comprise des deux côtés (lancer d'abord la cellule équivalente du notebook du jeu corrigé)
travail = pd.read_csv(dossier / "jeu_corrige_deux_populations.csv").set_index("ID")
if not any(c.startswith("decision | 5.") for c in travail.columns):
    display(Markdown("⚠️ La stratégie 5 manque côté jeu corrigé : lancer d'abord la dernière cellule de `essai_jeu_corrige_deux_populations.ipynb`."))
communs = travail.index.intersection(deux.index)
y_c = deux.loc[communs, "dpnm"].to_numpy()
lignes_c = []
for source, table in (("Codification de la banque (PAY_1 ≥ 2), jeu de base", deux), ("Règle métier (en incident à M, retard imminent), jeu corrigé", travail)):
    for s in [c.removeprefix("decision | ") for c in table.columns if c.startswith("decision | ")]:
        lignes_c.append({"Séparation": source, "Stratégie": s, **mesures(y_c, table.loc[communs, f"decision | {s}"].to_numpy(), table.loc[communs, f"note | {s}"].to_numpy())})
display(Markdown(f"**Les deux séparations face à face, sur les mêmes clients ({len(communs)}), stratégie 5 comprise**"))
display(pd.DataFrame(lignes_c).set_index(["Séparation", "Stratégie"]))


**Résultats accumulés, stratégie 5 comprise** (30000 clients, 6636 défauts, taux de défaut 22.1 %)

,Défauts détectés,Part des défauts (%),Clients signalés,Part des clients signalés (%),Précision des défauts prédits (%),Fausses alertes,Défauts manqués,Taux d'erreur,Ratio d'aire
Stratégie,,,,,,,,,
1. Règle de la banque seule (PAY_1 ≥ 2 en défaut),2177,32.8,3130,10.4,69.6,953,4459,0.180,0.308
2. Règle de la banque + ML pour les autres (CatBoost),2411,36.3,3606,12.0,66.9,1195,4225,0.181,0.564
3. Règle de la banque + ML pour les autres sans les PAY_n (CatBoost),2289,34.5,3376,11.3,67.8,1087,4347,0.181,0.553
"4. Deux ML (CatBoost pour PAY_1 ≥ 2, CatBoost pour les autres)",2411,36.3,3605,12.0,66.9,1194,4225,0.181,0.423
5. Un seul ML sur tous les clients (MLPClassifier),2430,36.6,3646,12.2,66.6,1216,4206,0.181,0.558


**Détail par population, stratégie 5 comprise**

Défauts détectés   sur  Fausses alertes  Précision (%)
Stratégie                                                            Population                                                        
1. Règle de la banque seule (PAY_1 ≥ 2 en défaut)                    PAY_1 ≥ 2               2177  2177              953           69.6
                                                                     PAY_1 < 2                  0  4459                0            NaN
2. Règle de la banque + ML pour les autres (CatBoost)                PAY_1 ≥ 2               2177  2177              953           69.6
                                                                     PAY_1 < 2                234  4459              242           49.2
3. Règle de la banque + ML pour les autres sans les PAY_n (CatBoost) PAY_1 ≥ 2               2177  2177              953           69.6
                                                                     PAY_1 < 2                112  4459              134           45.5
4. Deux ML (CatBoost pour PAY_1 ≥ 2, CatBoost pour les autres)       PAY_1 ≥ 2               2177  2177              952           69.6
                                                                     PAY_1 < 2                234  4459              242           49.2
5. Un seul ML sur tous les clients (MLPClassifier)                   PAY_1 ≥ 2               2086  2177              864           70.7
                                                                     PAY_1 < 2                344  4459              352           49.4

**Comparatif avec l'étude d'origine, stratégie 5 comprise** (fichier brut : même périmètre que l'étude)

Taux d'erreur  Ratio d'aire
Source             Méthode ou stratégie                                                                             
Yeh et Lien (2009) Réseau de neurones                                                            0.170         0.540
                   Arbre de classification                                                       0.170         0.536
                   Bayésien naïf                                                                 0.210         0.530
                   KNN                                                                           0.160         0.450
                   Régression logistique                                                         0.180         0.440
                   Analyse discriminante                                                         0.260         0.430
Cet essai          1. Règle de la banque seule (PAY_1 ≥ 2 en défaut)                             0.180         0.308
                   2. Règle de la banque + ML pour les autres (CatBoost)                         0.181         0.564
                   3. Règle de la banque + ML pour les autres sans les PAY_n (CatBoost)          0.181         0.553
                   4. Deux ML (CatBoost pour PAY_1 ≥ 2, CatBoost pour les autres)                0.181         0.423
                   5. Un seul ML sur tous les clients (MLPClassifier)                            0.181         0.558

**Les deux séparations face à face, sur les mêmes clients (28851), stratégie 5 comprise**

Défauts détectés  Part des défauts (%)  Clients signalés  Part des clients signalés (%)  \
Séparation                                                   Stratégie                                                                                                                                                                             
Codification de la banque (PAY_1 ≥ 2), jeu de base           1. Règle de la banque seule (PAY_1 ≥ 2 en défaut)                                                       2169                  34.6              3120                           10.8   
                                                             2. Règle de la banque + ML pour les autres (CatBoost)                                                   2403                  38.3              3596                           12.5   
                                                             3. Règle de la banque + ML pour les autres sans les PAY_n (CatBoost)                                    2279                  36.3              3363                           11.7   
                                                             4. Deux ML (CatBoost pour PAY_1 ≥ 2, CatBoost pour les autres)                                          2403                  38.3              3595                           12.5   
                                                             5. Un seul ML sur tous les clients (MLPClassifier)                                                      2418                  38.6              3626                           12.6   
Règle métier (en incident à M, retard imminent), jeu corrigé 1. Règle seule (en incident et retard imminent en défaut)                                               2871                  45.8              4743                           16.4   
                                                             2. Règle + ML autres clients (CatBoost)                                                                 2871                  45.8              4745                           16.4   
                                                             3. Règle + ML autres clients sans les PAY_n (CatBoost)                                                  2877                  45.9              4753                           16.5   
                                                             4. Trois ML (CatBoost en incident, MLPClassifier retard imminent, CatBoost autres clients)              2868                  45.7              4738                           16.4   
                                                             5. Un seul ML sur tous les clients (CatBoost)                                                           2527                  40.3              4010                           13.9   

                                                                                                                                                         Précision des défauts prédits (%)  Fausses alertes  Défauts manqués  Taux d'erreur  \
Séparation                                                   Stratégie                                                                                                                                                                        
Codification de la banque (PAY_1 ≥ 2), jeu de base           1. Règle de la banque seule (PAY_1 ≥ 2 en défaut)                                                                        69.5              951             4103          0.175   
                                                             2. Règle de la banque + ML pour les autres (CatBoost)                                                                    66.8             1193             3869          0.175   
                                                             3. Règle de la banque + ML pour les autres sans les PAY_n (CatBoost)                                                     67.8             1084             3993          0.176   
                             